In [ ]:
!pip install -q -U transformers
!pip install -q scikit-learn
!pip install -q sentence-transformers
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00


In [ ]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from google.colab import userdata
from huggingface_hub import login
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import Callable
import os
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM,AutoModelForCausalLM, AutoTokenizer, pipeline
import sys
import time
from typing import Callable
from sentence_transformers import CrossEncoder
from transformers import BitsAndBytesConfig
import json
import time
import gc


# **Setup HuggingFace and Game APIs**

## **HuggingFace**

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

## **Game APIs**

Let's import the client API folder from our GitHib repository

In [ ]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

Repository already present, update...
Already up to date.


Let's check if we are correctly logged in.

In [ ]:
API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Logged in as: GliEmbeddingRuspanti (role: student)


# **Model classes**

Here we build a model class so that we can easily define a model and implement as a method how the effective answering logic is implemented.

For instance we can generate a full response through a text-generation and then compute the answer of the model through similarity.

In [ ]:
class ModelFactory:
    """
    Base factory, subclasses create the pipeline
    create() produces Model instances sharing that pipeline.
    """
    def __init__(self, model_name: str, hf_token=None, device_map="cuda",
                 cache_dir=None, gen_args=None, quantization_config=None, trust_remote_code=True):
        self.model_name = model_name
        self.gen_args = gen_args or {}
        self._pipe = self._load_pipeline(
            model_name, hf_token, device_map, cache_dir, quantization_config, trust_remote_code
        )

    def _load_pipeline(self, model_name, hf_token, device_map, cache_dir, quantization_config, trust_remote_code):
        raise NotImplementedError

    def create(self, name: str, answer_fn: Callable, answers_in_question=True):
        """
        Produce a new Model instance sharing this factory's pipeline.
        Weights are not reloaded.
        """
        return HFPipelineModel(
            name=name,
            pipe=self._pipe,
            answer_fn=answer_fn,
            gen_args=self.gen_args,
            answers_in_question=answers_in_question,
        )


class HFCausalFactory(ModelFactory):
    """Factory for standard causal LMs (Llama, Phi, Qwen, ...)."""
    def _load_pipeline(self, model_name, hf_token, device_map: str = "cuda", cache_dir: str | None = None, quantization_config=None,trust_remote_code=False):
        model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map=device_map, torch_dtype="auto",
            trust_remote_code=trust_remote_code, token=hf_token, cache_dir=cache_dir,
            quantization_config=quantization_config,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
        return pipeline("text-generation", model=model, tokenizer=tokenizer)


class HFSeq2SeqFactory(ModelFactory):
    """Factory for encoder-decoder models (Flan-T5, ...)."""
    def _load_pipeline(self, model_name, hf_token, device_map: str = "cuda", cache_dir: str | None = None, quantization_config=None):
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name, device_map=device_map, torch_dtype="auto",
            trust_remote_code=True, token=hf_token, cache_dir=cache_dir,
            quantization_config=quantization_config,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
        return pipeline("text-generation", model=model, tokenizer=tokenizer)


class Model:
    """Base class. Subclasses implement generate().
       answer_fn decide how to get the final option."""
    def __init__(self, name: str, answer_fn: Callable):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        raise NotImplementedError

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        raw_output = self.generate(question, system_prompt)
        summary_answer, answer = self.answer_fn(raw_output, options)
        print(f"MODEL ANSWER ----->{raw_output}")
        return summary_answer, answer

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"


class HFPipelineModel(Model):
    """
    A Model that uses a shared pipeline injected by a ModelFactory.
    Never loads weights itself — that is the factory's responsibility.
    """
    DEFAULT_GEN_ARGS = {
        "max_new_tokens": 600,
        "return_full_text": False,
        "temperature": 0.5,
        "do_sample": True,
    }

    def __init__(self, name: str, pipe, answer_fn: Callable[[str, dict], str],
                 gen_args: dict = None, answers_in_question: bool = True):
        super().__init__(name, answer_fn)
        self._pipe = pipe
        self.gen_args = {**self.DEFAULT_GEN_ARGS, **(gen_args or {})}
        self.answers_in_question = answers_in_question

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        """Generates and process the answer through answer_fn."""

        if self.answers_in_question:
          # Converte le opzioni in plain text
          options_text = "\n".join(
              [f"- {value}" for value in options.values()]
          )

          question_full = f"{question}\n\nPossible options:\n{options_text}"
        else:
          question_full = question

        raw_output = self.generate(question_full, system_prompt)
        print(f"MODEL ANSWER ----->{raw_output}")
        start = time.perf_counter()
        summary_answer, answer = self.answer_fn(raw_output, options)
        print(f"ANSWER TIME: {time.perf_counter() - start}")
        return summary_answer, answer

    def generate(self, question: str, system_prompt: str = "") -> str:
        prompt = f"{system_prompt}\n\nQuestion: {question}"

        output = self._pipe(prompt, **self.gen_args)
        return output[0]["generated_text"]

# **Answers logic implementation through functions**

Here we'll implement the logic behind the true answer we'll give to the game.

## **TF-IDF + cosine similarity**

In [ ]:
# Strategy 2: TF-IDF + Cosine Similarity (Vector Space Model)

def pick_by_tfidf(model_output: str, options: dict):

    labels = list(options.keys())
    texts = [model_output] + [options[l] for l in labels]

    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(texts)

    query_vec = tfidf_matrix[0]
    option_vecs = tfidf_matrix[1:]

    # cosine similarities
    scores = cosine_similarity(query_vec, option_vecs)[0]

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = (
        float(best_score - np.mean(sorted_scores[1:]))
        if len(labels) > 1 else 0.0
    )

    # --- NORMALIZED MARGIN ---
    score_range = np.max(scores) - np.min(scores) + 1e-8

    normalized_margin = (
        (best_score - second_score) / score_range
        if len(labels) > 1 else 0.0
    )

    # --- probabilistic closeness of 2nd to 1st ---
    relative_second_closeness = (
        np.exp(second_score) /
        (np.exp(best_score) + np.exp(second_score))
        if len(labels) > 1 else 0.0
    )

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i])
        for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i])
        for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,
        "best_probability": best_prob,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

options_test = {
    'A': 'Un politico',
    'B': 'un personaggio televisivo',
    'C': 'qualcosa di assurdo',
    'D': 'Un calciatore'
}

model_response = "Napoleone era un grande leader politico e militare, imperatore dei francesi."

summary, best = pick_by_tfidf(model_response, options_test)

print("Picked option:", best)
print("summary:", summary)

Picked option: A
summary: {'best_option': 'A', 'best_score': 0.32858905992256504, 'best_probability': 0.3047505519470668, 'scores': {'A': 0.32858905992256504, 'B': 0.06962269809588174, 'C': 0.0, 'D': 0.09234056510965123}, 'softmax_probabilities': {'A': 0.3047505519470668, 'B': 0.23522140456331864, 'C': 0.21940174900740894, 'D': 0.24062629448220574}, 'gap_mean': 0.2746013055207207, 'relative_second_closeness': 0.4412110562772332, 'normalized_margin': 0.7189785553992648}


## **sBERT: A semantic similarity approach**

In [ ]:
# Strategy 3: Sentence-BERT (sBERT) Semantic Similarity

sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def pick_by_sbert(model_output: str, options: dict):

    labels = list(options.keys())
    all_texts = [model_output] + [options[l] for l in labels]

    embeddings = sbert_model.encode(
        all_texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_emb = embeddings[0]
    option_embs = embeddings[1:]

    # cosine similarity because embeddings are normalized
    scores = option_embs @ query_emb

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = (
        float(best_score - np.mean(sorted_scores[1:]))
        if len(labels) > 1 else 0.0
    )

    # --- NORMALIZED MARGIN ---
    score_range = np.max(scores) - np.min(scores) + 1e-8

    normalized_margin = (
        (best_score - second_score) / score_range
        if len(labels) > 1 else 0.0
    )

    # --- probabilistic closeness of 2nd to 1st ---
    relative_second_closeness = (
        np.exp(second_score) /
        (np.exp(best_score) + np.exp(second_score))
        if len(labels) > 1 else 0.0
    )

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i])
        for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i])
        for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,
        "best_probability": best_prob,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

options_test = {
    'A': 'A politician',
    'B': 'a television personality',
    'C': 'something absurd',
    'D': 'a football player'
}

model_response = (
    "Napoleon was a great political and military leader, "
    "Emperor of the French."
)

summary, best = pick_by_sbert(model_response, options_test)

print("Picked option:", best)
print("Summary:", summary)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Picked option: A
Summary: {'best_option': 'A', 'best_score': 0.3144106864929199, 'best_probability': 0.3078668713569641, 'scores': {'A': 0.3144106864929199, 'B': 0.006208924576640129, 'C': -0.07364878058433533, 'D': 0.13410882651805878}, 'softmax_probabilities': {'A': 0.3078668713569641, 'B': 0.22621044516563416, 'C': 0.2088482528924942, 'D': 0.25707441568374634}, 'gap_mean': 0.2921876907348633, 'relative_second_closeness': 0.45504625162805523, 'normalized_margin': 0.4646243155002594}


## **Cross encoder**

In [ ]:
modelCrossencoder = CrossEncoder('cross-encoder/stsb-distilroberta-base', trust_remote_code=True)

'''
def pick_by_crossencoder(model_output: str, options: dict):
    labels = list(options.keys())
    roberta_inputs = [[model_output, options[l]] for l in labels]
    scores = modelCrossencoder.predict(roberta_inputs)
    best_label = labels[int(np.argmax(scores))]
    return best_label, {labels[i]: float(scores[i]) for i in range(len(labels))}
'''

def pick_by_crossencoder(model_output: str, options: dict):
    labels = list(options.keys())
    pairs = [[model_output, options[l]] for l in labels]

    scores = modelCrossencoder.predict(pairs)

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = float(best_score - np.mean(sorted_scores[1:])) if len(labels) > 1 else 0.0

    # --- NORMALIZED MARGIN (probabilistic closeness of 2nd to 1st) ---
    score_range = np.max(scores) - np.min(scores) + 1e-8
    normalized_margin = (best_score - second_score) / score_range

    # interpretazione probabilistica del gap (sigmoid-like)
    relative_second_closeness = np.exp(second_score) / (np.exp(best_score) + np.exp(second_score))

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i]) for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i]) for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

options_test = {
    'A': 'a television personality',
    'B': 'A politician',
    'C': 'something absurd',
    'D': 'a football player'
}

model_response = (
    "Napoleon was a great political and military leader, "
    "Emperor of the French."
)

summary, best = pick_by_crossencoder(model_response, options_test)

print("Picked option:", best)
print("Summary", summary)

config.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

Picked option: B
Summary {'best_option': 'B', 'best_score': 0.2227260023355484, 'scores': {'A': 0.028937358409166336, 'B': 0.2227260023355484, 'C': 0.0470624640583992, 'D': 0.014161717146635056}, 'softmax_probabilities': {'A': 0.23710934817790985, 'B': 0.2878127694129944, 'C': 0.24144618213176727, 'D': 0.2336316853761673}, 'gap_mean': 0.19267214834690094, 'relative_second_closeness': 0.45619669656547107, 'normalized_margin': 0.842251181602478}


## **HuggingFace**

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

## **Game APIs**

Let's import the client API folder from our GitHib repository

In [ ]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

Repository already present, update...
Already up to date.


Let's check if we are correctly logged in.

In [ ]:
API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Logged in as: GliEmbeddingRuspanti (role: student)


# **Models**

Let's define a set of models we want to use.

Specifically we'll embed models in a Model class object. We'll specify the aswering logic through one of the previous defined classes.

In the end we'll make a full set so that we can easily test all the models and have benchmarks.

Each factory loads weights exactly once

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

##**Models with Llama 3B**

In [ ]:
llama_factory = HFCausalFactory(
    model_name="meta-llama/Llama-3.2-3B-Instruct",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
)

llama_sbert        = llama_factory.create("llama-sbert",        pick_by_sbert)
llama_tf_idf       = llama_factory.create("llama-tfidf",        pick_by_tfidf)
llama_crossencoder = llama_factory.create("llama-crossencoder", pick_by_crossencoder)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
del llama_factory
del llama_sbert
del llama_tf_idf
del llama_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with Phi-3.5**

In [ ]:
phi_factory = HFCausalFactory(
    model_name="microsoft/Phi-3.5-mini-instruct",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    trust_remote_code=False,
)

phi_sbert          = phi_factory.create("phi-sbert",            pick_by_sbert)
phi_tf_idf         = phi_factory.create("phi-tfidf",            pick_by_tfidf)
phi_crossencoder   = phi_factory.create("phi-crossencoder",     pick_by_crossencoder)

config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [ ]:
del phi_factory
del phi_sbert
del phi_tf_idf
del phi_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with Llama 8B**

In [ ]:
llama8b_factory = HFCausalFactory(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    gen_args={
        "max_new_tokens": 100,
        "return_full_text": False,
        "temperature": 0.3,
        "do_sample": True,
    }
)

llama8b_sbert        = llama8b_factory.create("llama8b-sbert",        pick_by_sbert)
llama8b_tf_idf       = llama8b_factory.create("llama8b-tfidf",        pick_by_tfidf)
llama8b_crossencoder = llama8b_factory.create("llama8b-crossencoder", pick_by_crossencoder)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
del llama8b_factory
del llama8b_sbert
del llama8b_tf_idf
del llama8b_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with Gemma-2**

In [ ]:
gemma_factory = HFCausalFactory(
    model_name="google/gemma-2-9b-it",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    trust_remote_code=False,
    gen_args={
        "max_new_tokens": 100,   # era 600 — Gemma 9B genera ~15tok/s, 100 tok = ~7s
        "return_full_text": False,
        "temperature": 0.3,      # più basso = risposte più brevi e dirette
        "do_sample": True,
    }
)

gemma_sbert        = gemma_factory.create("gemma2-9b-sbert",        pick_by_sbert)
gemma_tf_idf       = gemma_factory.create("gemma2-9b-tfidf",        pick_by_tfidf)
gemma_crossencoder = gemma_factory.create("gemma2-9b-crossencoder", pick_by_crossencoder)

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [ ]:
del gemma_factory
del gemma_sbert
del gemma_tf_idf
del gemma_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with DeepSeek Llama-8B**

In [ ]:
# DeepSeek R1 Distill 8B — thinking model basato su Llama 3
deepseek_factory = HFCausalFactory(
    model_name="deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    gen_args={
        "max_new_tokens": 200,   # più alto degli altri: il thinking token <think> occupa spazio
        "return_full_text": False,
        "temperature": 0.6,      # valore raccomandato da DeepSeek per i modelli R1
        "do_sample": True,
    }
)

deepseek_sbert        = deepseek_factory.create("deepseek-r1-sbert",        pick_by_sbert)
deepseek_tf_idf       = deepseek_factory.create("deepseek-r1-tfidf",        pick_by_tfidf)
deepseek_crossencoder = deepseek_factory.create("deepseek-r1-crossencoder", pick_by_crossencoder)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
del deepseek_factory
del deepseek_sbert
del deepseek_tf_idf
del deepseek_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with DeepSeek Qwen-7B**

In [ ]:
# DeepSeek R1 Distill 7B — thinking model basato su Qwen
deepseek_factory = HFCausalFactory(
    model_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    hf_token=HF_TOKEN,
    cache_dir="./models_cache",
    quantization_config=bnb_config,
    gen_args={
        "max_new_tokens": 200,   # i token di reasoning occupano spazio
        "return_full_text": False,
        "temperature": 0.6,      # raccomandato per R1
        "do_sample": True,
    }
)

deepseek_sbert        = deepseek_factory.create("deepseek-r1-qwen-sbert",        pick_by_sbert)
deepseek_tf_idf       = deepseek_factory.create("deepseek-r1-qwen-tfidf",        pick_by_tfidf)
deepseek_crossencoder = deepseek_factory.create("deepseek-r1-qwen-crossencoder", pick_by_crossencoder)

In [ ]:
del deepseek_factory
del deepseek_sbert
del deepseek_tf_idf
del deepseek_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with Qwen2.5**

In [ ]:
qwen_factory = HFCausalFactory(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    gen_args={
        "max_new_tokens": 100,
        "return_full_text": False,
        "temperature": 0.3,
        "do_sample": True,
    }
)

qwen_sbert        = qwen_factory.create("qwen2.5-7b-sbert",        pick_by_sbert)
qwen_tf_idf       = qwen_factory.create("qwen2.5-7b-tfidf",        pick_by_tfidf)
qwen_crossencoder = qwen_factory.create("qwen2.5-7b-crossencoder", pick_by_crossencoder)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
del qwen_factory
del qwen_sbert
del qwen_tf_idf
del qwen_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models initialization**

In [ ]:

           """
            You will receive an input string. Your task is to rewrite it while preserving the original meaning, but removing all unnecessary content.

            Rules:
            - Keep only essential information.
            - Remove redundancy, filler words, repetitions, and anything superfluous.
            - Simplify sentence structure without changing meaning.
            - Do not add new information or interpretations.
            - Do not alter the original meaning.
            - If the text is already concise and essential, return it unchanged.

            context: {{QUESTION}}
            INPUT: {{TRANSCRIPT}}

            The output MUST be the input sentence without all the meaningless inormation.
            You are allowed to reshape the output in order to let it become clearer.
            You have to look at the information contained in the 'context' to understand what has meaning.
            If you are unsure about the output leave the sentence as it is.

            Return ONLY the output sentence without any other text.
          """

In [ ]:
prompts = {
    "robust_prompt": """
        You are a highly skilled competitive quiz player. Your primary objective is to maximize answer accuracy and provide the most factually correct response possible for every question.

        Behavior rules:
        - Always try to determine the correct answer using reasoning, world knowledge, context clues, and inference.
        - If the answer is uncertain, provide the most probable answer rather than refusing to answer.
        - Avoid random guessing when possible; make educated inferences instead.
        - Prefer concise, direct answers over long explanations.
        - Do not roleplay, joke, or add unnecessary commentary.
        - Do not intentionally hedge unless uncertainty is genuinely high.
        - Use careful internal reasoning before answering.
        - If multiple answers seem possible, choose the one most likely to be accepted in a standard quiz context.
        - Prioritize commonly accepted canonical answers.
        - Be robust to ambiguous wording and infer likely intent.
        - Optimize for correctness over creativity or personality.

        Output rules:
        - Respond with only the final answer.
        - Do not explain your reasoning unless explicitly requested.
        - Keep answers short and precise.
        - The answer is one of the possible options listed
    """,

    "exhaustive_prompt": "You are a quiz game expert. Answer in the most exhaustive manner.",

    "zero_shot_prompt": """You are a quiz expert. You will be given a multiple choice question.
Choose the correct answer from the options provided.
Reply with only the text of the correct answer, nothing else.""",

    "one_shot_prompt": """You are a quiz expert. You will be given a multiple choice question.
Choose the correct answer from the options provided.
Reply with only the text of the correct answer, nothing else.

Example:
Question: What is the capital of France?
Options:
- Berlin
- Madrid
- Paris
- Rome

Answer: Paris

Now answer the following question.""",
}

In [ ]:
#models = []
'''
llama_sbert, llama_tf_idf, llama_crossencoder,
phi_sbert, phi_tf_idf, phi_crossencoder,
llama8b_sbert, llama8b_tf_idf, llama8b_crossencoder,
gemma_sbert, gemma_tf_idf, gemma_crossencoder,
deepseek_sbert, deepseek_tf_idf, deepseek_crossencoder,
'''
models = [gemma_sbert]

# **The Game**

In [ ]:
def answer_ensemble(models: list, question_text: str, options: dict, system_prompt: str, verbose=False):
    """
    Chiama model.answer su ogni modello dell'array e restituisce
    la risposta con la best_score più alta tra tutti i modelli.
    """
    best_answer = None
    best_summary = None
    best_score = -float('inf')

    for model in models:
        summary, answer = model.answer(question_text, options, system_prompt)
        score = summary.get("best_score", 0.0) if isinstance(summary, dict) else 0.0

        if verbose:
            print(f"  [{model.name}] answer={answer} | best_score={score:.4f}")

        if score > best_score:
            best_score = score
            best_answer = answer
            best_summary = summary

    if verbose:
        print(f"  => Ensemble winner: answer={best_answer} | score={best_score:.4f}")

    return best_summary, best_answer

In [ ]:
def play_game(game, models_config, verbose=False):
    models = models_config["models"]
    system_prompt = models_config["system_prompt"]

    log = []

    while game.in_progress:
        question = game.current_question
        if not question:
            print("No question available. Game may have ended.")
            break

        print(f"\n--- Level {game.current_level} ---")
        print(f"Q: {question.text}")
        for opt in question.options:
            print(f"  [{opt.id}] {opt.text}")

        time_left = game.time_remaining
        if time_left:
            print(f"\nTime remaining: {time_left:.1f}s")

        options = {f"{opt.id}": opt.text for opt in question.options}
        #print(f"Options: {options}")
        t0 = time.time()
        answer_summary, answer_input = answer_ensemble(
            models, question.text, options, system_prompt, verbose=verbose
        )
        inference_time = time.time() - t0

        print(f"Ensemble answer: {answer_input}")
        answer_id = int(answer_input)
        choosen_answer = question.options[answer_id]

        result = game.answer(answer_id)

        if result.correct:
            print(" CORRECT!")
            if result.game_over:
                print(f"\n CONGRATULATIONS! You completed the game!")
                print(f" Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f" Earned so far: ${result.earned_amount:,.2f}")
        elif result.timed_out:
            print("TIMED OUT!")
            print(f"\n Game Over! | Final earnings: ${result.earned_amount:,.2f}")
        elif not result.correct:
            print(" WRONG ANSWER!")
            print(f"\n Game Over! | Final earnings: ${result.earned_amount:,.2f}")
        # save outcome in the log(useful for graphs)
        entry = {
            'level'          : game.current_level,
            'question'       : question.text,
            'options'        : question.options,
            'chosen_option'  : choosen_answer.text,
            'correct'        : result.correct,
            'timed_out'      : result.timed_out,
            'inference_time' : round(inference_time, 2),
            'answer_summary' : answer_summary,
        }
        log.append(entry)

    ensemble_name = " + ".join(m.name for m in models)

    summary = {
        'model'          : ensemble_name,
        'final_level'    : game.current_level,
        'earned_amount'  : game.earned_amount,
        'num_questions'  : len(log),
        'num_correct'    : sum(1 for e in log if e['correct']),
        'num_timed_out'  : sum(1 for e in log if e['timed_out']),
        'avg_inference_s': round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'            : log,
    }

    print(f"\n=== Game Summary ===")
    print(f"Ensemble  : {ensemble_name}")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

    return summary

In [ ]:
COMPETITIONS = [0]
"[0, 1, 2, 3, 4, 5]"
COMPETITION_NAMES = {
    0: "Entertainment",
    1: "Ancient History and Politics",
    2: "Science and Nature",
    3: "Maths",
    4: "Philosophy and Psychology",
    5: "News",
}
NUM_RUNS = 10

def run_analysis(models: list, prompts: list, client, num_runs: int = NUM_RUNS, results_file: str = "/content/analysis_results.json"):
    """
    Per ogni modello nell'array e per ogni prompt nell'array,
    gioca num_runs partite e salva i risultati aggregati.

    Struttura risultato:
    {
        "llama-sbert": {
            "prompt_0": {
                "comp_0": [ {correct, confidence_array, inference_times}, ... ],  # num_runs entries
                "comp_1": [ ... ],
                ...
            },
            "prompt_1": { ... }
        },
        ...
    }

    Dopo ogni modello salva su file, così se il runtime crasha non si perde tutto.
    """

    # Carica risultati esistenti se il file esiste già (resume da crash)
    if os.path.exists(results_file):
        with open(results_file, 'r') as f:
            results = json.load(f)
        print(f"Loaded existing results from {results_file}")
    else:
        results = {}

    for model in models:
        model_name = model.name
        print(f"\n{'='*70}")
        print(f"MODEL: {model_name}")
        print(f"{'='*70}")

        if model_name not in results:
            results[model_name] = {}

        for prompt_name, system_prompt in prompts.items():
            prompt_key = prompt_name
            print(f"\n  --- Prompt: {prompt_name} ---")

            if prompt_key not in results[model_name]:
                results[model_name][prompt_key] = {}
            for comp_id in COMPETITIONS:
                comp_name = COMPETITION_NAMES[comp_id]
                comp_key = f"comp_{comp_id}_{comp_name.replace(' ', '_')}"  # es. "comp_0_Entertainment"
                print(f"\n    Competition {comp_id} — {comp_name}")

                if comp_key not in results[model_name][prompt_key]:
                    results[model_name][prompt_key][comp_key] = []

                # Calcola quante run mancano (resume da crash)
                runs_done = len(results[model_name][prompt_key][comp_key])
                runs_left = num_runs - runs_done
                if runs_left <= 0:
                    print(f"    Already completed {num_runs} runs, skipping.")
                    continue

                print(f"    Running {runs_left} games (already done: {runs_done})")

                for run_idx in range(runs_left):
                    print(f"    Run {runs_done + run_idx + 1}/{num_runs}...")

                    models_config = {
                        "models": [model],
                        "system_prompt": system_prompt,
                    }

                    game = client.game.start(competition_id=comp_id)
                    summary = play_game(game, models_config, verbose=False)

                    # Estrai solo i dati rilevanti per l'analisi
                    run_data = {
                        "prompt_name"     : prompt_name,
                        "competition_name": comp_name,
                        "correct"         : summary["num_correct"],
                        "total_questions" : summary["num_questions"],
                        "timed_out"       : summary["num_timed_out"],
                        "earned"          : summary["earned_amount"],
                        "final_level"     : summary["final_level"],
                        "avg_inference_s" : summary["avg_inference_s"],
                        "inference_times" : [e["inference_time"] for e in summary["log"]],
                        "confidence_array": [
                            e["answer_summary"].get("best_score", None)
                            if isinstance(e.get("answer_summary"), dict) else None
                            for e in summary["log"]
                        ],
                        "correctness_array": [e["correct"] for e in summary["log"]],
                    }

                    results[model_name][prompt_key][comp_key].append(run_data)

        # Salva su file dopo ogni modello — così si può liberare memoria senza perdere i dati
        with open(results_file, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\n  Results saved to {results_file} after model {model_name}")

    return results

In [ ]:
# Esegui l'analisi per i modelli attualmente in memoria
# Cambia questa lista in base a quali modelli sono caricati

analysis_results = run_analysis(
    models=models,
    prompts=prompts,
    client=client,
    num_runs=NUM_RUNS,
)

Loaded existing results from /content/analysis_results.json

MODEL: gemma2-9b-sbert

  --- Prompt: robust_prompt ---

    Competition 0 — Entertainment
    Running 10 games (already done: 0)
    Run 1/10...


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Level 1 ---
Q: Who is considered by Paul McCartney to be the 'fifth Beatle' in his 2016 memorial post?
  [0] George Martin
  [1] Ringo Starr
  [2] John Lennon
  [3] Brian Epstein

Time remaining: 29.9s
MODEL ANSWER ----->


Answer:
George Martin 

ANSWER TIME: 0.013784871999632742
Ensemble answer: 0
 WRONG ANSWER!

 Game Over! | Final earnings: $0.00

=== Game Summary ===
Ensemble  : gemma2-9b-sbert
Reached Level: 1
Total Earnings: $0.00
    Run 2/10...


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Level 1 ---
Q: What is the primary principle that influenced Drake's incorporation of R&B into his hip-hop music?
  [0] His mother's encouragement
  [1] His interest in Caribbean dancehall
  [2] His father's influence
  [3] His exposure to UK drill

Time remaining: 29.9s
MODEL ANSWER -----> music
- His admiration for R&B vocalists



Answer: 
His admiration for R&B vocalists 

ANSWER TIME: 0.013156571000763506
Ensemble answer: 1
 WRONG ANSWER!

 Game Over! | Final earnings: $0.00

=== Game Summary ===
Ensemble  : gemma2-9b-sbert
Reached Level: 1
Total Earnings: $0.00
    Run 3/10...


...

 WRONG ANSWER!

 Game Over! | Final earnings: $100.00

=== Game Summary ===
Ensemble  : gemma2-9b-sbert
Reached Level: 2
Total Earnings: $100.00
    Run 7/10...

--- Level 1 ---
Q: What is the primary principle that influenced Drake's incorporation of R&B into his hip-hop music?
  [0] His mother's encouragement
  [1] His father's influence
  [2] His exposure to UK drill
  [3] His interest in Caribbean dancehall

Time remaining: 29.9s
MODEL ANSWER ----->

**Answer:**

The primary principle that influenced Drake's incorporation of R&B into his hip-hop music is **his mother's encouragement**.  

Here's a breakdown of why:

* **Drake's Mother's Background:** Drake's mother, Sandi Graham, is a devout Christian and a former teacher. She exposed Drake to a wide range of musical genres, including R&B, soul, and gospel, from a young age. This early exposure played
ANSWER TIME: 0.02046801300093648
Ensemble answer: 0


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 WRONG ANSWER!

 Game Over! | Final earnings: $0.00

=== Game Summary ===
Ensemble  : gemma2-9b-sbert
Reached Level: 1
Total Earnings: $0.00
    Run 8/10...

--- Level 1 ---
Q: Which of the following best describes the connection between Drake's personal life and his music?
  [0] His music is primarily about political commentary
  [1] His music focuses solely on fictional stories
  [2] His music often reflects personal experiences and emotions
  [3] His music avoids mentioning personal life

Time remaining: 29.9s
MODEL ANSWER -----> altogether

**Answer:**

The connection between Drake's personal life and his music is deeply intertwined.  The answer that best describes this relationship is **"His music often reflects personal experiences and emotions."**

Here's a breakdown of why:

* **Drake's music is known for its vulnerability and honesty.** He frequently delves into themes of love, heartbreak, success, fame, and the complexities of relationships. 

* **Many of his songs are inspir

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
Q: When did Amazon Prime Video first announce its intention to produce The Boys as a television series?
  [0] 2018
  [1] 2008
  [2] 2020
  [3] 2016

Time remaining: 29.9s


##**Print of results**

##RAG + AGENTIC MATH OUTPUT

In [ ]:
def print_results(results):
    for model_name, competitions in results.items():

        print("\n" + "=" * 80)
        print(f"MODELLO: {model_name}")
        print("=" * 80)

        for i, summary in enumerate(competitions):

            print(f"\n🏁 Competition {i}")
            print("-" * 60)

            print(f"Model name        : {summary['model']}")
            print(f"Final level       : {summary['final_level']}")
            print(f"Earned amount     : €{summary['earned_amount']}")
            print(f"Questions         : {summary['num_questions']}")
            print(f"Correct answers   : {summary['num_correct']}")
            print(f"Timed out         : {summary['num_timed_out']}")
            print(f"Avg inference     : {summary['avg_inference_s']} s")

            accuracy = (
                summary['num_correct'] / summary['num_questions'] * 100
                if summary['num_questions'] > 0 else 0
            )

            print(f"Accuracy          : {accuracy:.1f}%")

            print("\n📋 Question Log")
            print("-" * 60)

            confidence_array = []

            for q_idx, entry in enumerate(summary['log'], start=1):

                status = "✅" if entry['correct'] else "❌"

                if entry.get('timed_out'):
                    status = "⏰"

                # ─────────────────────────────────────────────
                # CONFIDENCE EXTRACTION (robust fallback chain)
                # ─────────────────────────────────────────────
                answer_summary = entry.get("answer_summary", {})

                if isinstance(answer_summary, dict):
                    if "normalized_margin" in answer_summary:
                        conf = answer_summary["normalized_margin"]

                    elif "confidence" in answer_summary:
                        conf = answer_summary["confidence"]

                    else:
                        conf = None
                else:
                    conf = None

                confidence_array.append(conf)

                print(
                    f"{q_idx:02d}. "
                    f"{status} "
                    f"Time: {entry['inference_time']:.2f}s "
                    f"Conf: {conf if conf is not None else 'N/A'}"
                )

            # ─────────────────────────────────────────────
            # PRINT SUMMARY CONFIDENCE ARRAY
            # ─────────────────────────────────────────────
            print("\n📊 Confidence Array:")
            if any(c is not None for c in confidence_array):
                print(confidence_array)
            else:
                print("Confidence not available")

        print("\n")

# final print
print_results(results)


MODELLO: qwen_deepseek_cross

🏁 Competition 0
------------------------------------------------------------
Model name        : qwen2.5-7b-crossencoder + deepseek-r1-crossencoder
Final level       : 1
Earned amount     : €0
Questions         : 1
Correct answers   : 0
Timed out         : 1
Avg inference     : 36.47 s
Accuracy          : 0.0%

📋 Question Log
------------------------------------------------------------
01. ⏰ Time: 36.47s Conf: 0.7478112578392029

📊 Confidence Array:
[0.7478112578392029]

🏁 Competition 1
------------------------------------------------------------
Model name        : qwen2.5-7b-crossencoder + deepseek-r1-crossencoder
Final level       : 1
Earned amount     : €0
Questions         : 1
Correct answers   : 0
Timed out         : 1
Avg inference     : 33.51 s
Accuracy          : 0.0%

📋 Question Log
------------------------------------------------------------
01. ⏰ Time: 33.51s Conf: 0.7657882571220398

📊 Confidence Array:
[0.7657882571220398]

🏁 Competition 2
-

In [ ]:
results = {}

for model_name, config in models.items():
    print(f"\n########## MODEL: {model_name} ##########")

    model = config["model"]
    system_prompt = config["system_prompt"]

    model_results = []
    complete_log = []

    for comp_id in [3]:
        print(f"\n--- Competition {comp_id} ---")

        game = client.game.start(competition_id=comp_id)

        summary, competition_log = play_game(game, model, system_prompt)
        complete_log += competition_log

        model_results.append(summary)

    results[model_name] = model_results

    print("========SAVING FULL GAME=======")
    game_log = {
        "session_id": game.session_id,
        "competition": game.state.competition.id,
        "questions_log": complete_log
    }

    # Salvi l'intera giocata su una singola riga del JSONL
    with open("/content/drive/MyDrive/Progetto-NLP/Branch-rag/log_games.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(game_log) + "\n")


########## MODEL: RAG ##########

--- Competition 3 ---

--- Level 1 ---
Q: Zoey is laying bricks for her patio. The salesman wants to sell Zoey as many bricks as possible to cover her patio with a thickness of one brick, while not having any extra bricks. The patio area is a rectangle with dimensions 12 feet by 10 feet, and each individual brick is 4 inches by 6 inches by 2 inches. What would be the greatest number of bricks the salesman could sell to meet his sales criteria?

  [0] 2,880
  [1] 2,160
  [2] 5,760
  [3] 1,440

Time remaining: 29.9s
QUESTION:  Zoey is laying bricks for her patio. The salesman wants to sell Zoey as many bricks as possible to cover her patio with a thickness of one brick, while not having any extra bricks. The patio area is a rectangle with dimensions 12 feet by 10 feet, and each individual brick is 4 inches by 6 inches by 2 inches. What would be the greatest number of bricks the salesman could sell to meet his sales criteria?
OPTIONS:  {'0': '2,880', '1'

In [ ]:
gc.collect()
torch.cuda.empty_cache()